Daily Challenge : Building Trustworthy Insights with BERT

O. Importing Libraries

In [258]:
import datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,Trainer
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score


1. Data Loading & Inspection

In [71]:
dataset = datasets.load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [72]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [73]:
dataset['train'].num_rows
def print_split_distrib(split):
    print(f'--- Dataset split: {split} ---')
    class_distrib = [0, 0, 0]

    for i in range (0, dataset[split].num_rows):
        i_label = dataset[split][i]['label']
        class_distrib[i_label] += 1

    print(class_distrib)

    for number in range(0, 3):
        print(f'Label {number}: Distribution {class_distrib[number]/dataset[split].num_rows*100}')

In [74]:
print_split_distrib('train')
print_split_distrib('test')
print_split_distrib('validation')

--- Dataset split: train ---
[7093, 20673, 17849]
Label 0: Distribution 15.549709525375425
Label 1: Distribution 45.320618217691546
Label 2: Distribution 39.12967225693303
--- Dataset split: test ---
[3972, 5937, 2375]
Label 0: Distribution 32.33474438293715
Label 1: Distribution 48.331162487788994
Label 2: Distribution 19.334093129273853
--- Dataset split: validation ---
[312, 869, 819]
Label 0: Distribution 15.6
Label 1: Distribution 43.45
Label 2: Distribution 40.949999999999996


TODO:Save two example tweets pe label for later visualizatin.

2. Tokenizatin Pipeline

In [75]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [76]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def preprocess(batch):
  enc = tokenizer(batch['text'], max_length=128, padding='max_length', truncation=True)
  enc['labels'] = batch['label']
  return enc

dataset_enc = dataset.map(preprocess, batched=True)

dataset_enc

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [77]:
dataset_enc.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

TODD: Shuffle the items

3. Fine-Tuning Setup

In [78]:
#Instanciating our model (3 labels)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

training_args = TrainingArguments(num_train_epochs=3, per_device_eval_batch_size=32, learning_rate=5e-5, weight_decay=0.01, load_best_model_at_end=True, eval_strategy='epoch', save_strategy='epoch')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [229]:
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Correctly assign 'cpu' if CUDA is not available
print(f"Using device: {device}")

if device == 'cuda':
    # It's crucial to restart the Colab runtime after installing CUDA-enabled torch
    # If you see 'AssertionError: Torch not compiled with CUDA enabled' here,
    # please ensure you have restarted the runtime after running cell 215fcc33.
    print("CUDA est disponible ! Vous pouvez utiliser votre GPU.")
    try:
        print(f"Nom du périphérique CUDA : {torch.cuda.get_device_name(0)}")
    except AssertionError:
        print("Attention: CUDA est disponible, mais PyTorch indique qu'il n'est pas compilé avec CUDA activé.\nCela signifie généralement qu'un redémarrage de l'environnement d'exécution est nécessaire après l'installation des paquets compatibles CUDA.")
    # Make sure 'model' is defined and loaded before this cell if you intend to use it
    # model.to(device) # Uncomment this line if 'model' is already defined
else:
    print("CUDA n'est pas disponible. Exécution sur CPU.")

Using device: cpu
CUDA n'est pas disponible. Exécution sur CPU.


In [243]:
print(f"Version de torch installée: {torch.__version__}")

Version de torch installée: 2.11.0+cpu


In [236]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  prediction = np.argmax(logits, axis=-1)
  accuracy = accuracy_score(y_true=labels, y_pred=prediction)
  macro_f1 = f1_score(y_true=labels, y_pred=prediction, average='macro')
  return {'accuracy': accuracy, 'macro_f1': macro_f1}

In [234]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_enc ['train'],
    eval_dataset=dataset_enc['test'],
   )


In [224]:
# Uninstall existing torch, torchvision, and torchaudio
# This addresses the 'VideoReader' import error by removing torchvision if it's not needed for text tasks.
!pip uninstall -y torch torchvision torchaudio

# Install only torch with CUDA 12.1 support
!pip install torch==2.2.0 --index-url https://download.pytorch.org/whl/cu121

print("Torch (without torchvision/torchaudio) reinstalled. Please restart the runtime now.")

Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 28.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 28.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 35.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB

Torch (without torchvision/torchaudio) reinstalled. Please restart the runtime now.


In [237]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  prediction = np.argmax(logits, axis=-1)
  accuracy = accuracy_score(y_true=labels, y_pred=prediction)
  macro_f1 = f1_score(y_true=labels, y_pred=prediction, average='macro')
  return {'accuracy': accuracy, 'macro_f1': macro_f1}

4. Evaluation & Calibration

In [262]:
import torch

print(f"Version de torch installée: {torch.__version__}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if device == 'cuda':
    print("CUDA est disponible ! Vous pouvez utiliser votre GPU.")
    try:
        print(f"Nom du périphérique CUDA : {torch.cuda.get_device_name(0)}")
    except AssertionError:
        print("Attention: CUDA est disponible, mais PyTorch indique qu'il n'est pas compilé avec CUDA activé.\nCela signifie généralement qu'un redémarrage de l'environnement d'exécution est nécessaire après l'installation des paquets compatibles CUDA.")
else:
    print("CUDA n'est pas disponible. Exécution sur CPU.")

Version de torch installée: 2.11.0+cpu
Using device: cpu
CUDA n'est pas disponible. Exécution sur CPU.


In [265]:
checkpoint = "distilbert-base-uncased"
base_model = AutoModel.from_pretrained(checkpoint, output_attentions=True)
base_model.to(model.device)
base_model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [271]:
# Get a single sample from the test set to generate inputs
# Temporarily retrieve the sample as a dictionary to avoid the torchvision import issue
with dataset_enc.formatted_as(type='python'):
    sample_batch = dataset_enc['test'][0]

inputs = {
    'input_ids': torch.tensor(sample_batch['input_ids']).unsqueeze(0).to(model.device), # Add batch dimension
    'attention_mask': torch.tensor(sample_batch['attention_mask']).unsqueeze(0).to(model.device)
}

with torch.no_grad():
    outputs = base_model(**inputs)

attentions = outputs.attentions  # tuple (layers)

last_layer_attn = attentions[-1]  # (batch, heads, seq_len, seq_len)

attn = last_layer_attn[0]         # remove batch
avg_attn = attn.mean(dim=0)      # average over heads